<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/notebooks/03_rlm_and_more_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers bitsandbytes accelerate rich psutil \
    sentence-transformers faiss-cpu rouge-score scikit-learn

print("all dependencies installed")

all dependencies installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import psutil
import time
from rich.console import Console
from rich.table import Table
import re
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

console = Console()
console.print("imports successful")

imports successful

In [3]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

console.print(f"Loading {model_name} in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
console.print("model loaded")

Loading Qwen/Qwen2.5-3B-Instruct in 4-bit...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model loaded

In [4]:
def profile():
    ram = psutil.virtual_memory().used / 1e9
    console.print(f"RAM Used: {ram:.2f} GB")

profile()

RAM Used: 4.06 GB

In [5]:
def ask(prompt, max_new_tokens=100, temperature=0.3):
    """Generate response from model."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4000).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

response = ask("Who is Devin Booker?")
console.print(f"Test Answer: {response}")

Test Answer: Who is Devin Booker? Devin Booker is an American professional basketball player who plays as a 
shooting guard for the Phoenix Suns of the National Basketball Association (NBA). He was drafted by the Suns with 
the 15th overall pick in the 2017 NBA draft. Booker has become known for his exceptional shooting ability, 
particularly from beyond the three-point line, and he is considered one of the best shooters in the league.

Key aspects of Devin Booker's career include:

1. Consistent scoring: Booker is known

In [6]:
print("\nLoading text from Project Gutenberg...")

import urllib.request

url = "https://www.gutenberg.org/files/1727/1727-0.txt"

try:
    with urllib.request.urlopen(url) as response:
        long_context = response.read().decode('utf-8')

    start_marker = "*** START OF"
    end_marker = "*** END OF"

    if start_marker in long_context:
        long_context = long_context.split(start_marker)[1]
    if end_marker in long_context:
        long_context = long_context.split(end_marker)[0]

    long_context = long_context * 2

    print("Downloaded The Odyssey")
    print(f"Words: {len(long_context.split()):,}")

except Exception as e:
    print(f"Failed to download: {e}")
    long_context = """Machine learning text""" * 1000

test_queries = [
    "What is the Telemachy?",
    "Who is Polyphemos?",
    "What happened to Odysseus during his wanderings?"
]

reference_answers = [
    "The Telemachy is the first four books of the Odyssey focusing on Telemachos.",
    "Polyphemos is a Cyclops, son of Poseidon, who was blinded by Odysseus.",
    "Odysseus wandered for years after the Trojan War facing various challenges."
]

print(f"Created {len(test_queries)} test queries")


Loading text from Project Gutenberg...
Downloaded The Odyssey
Words: 259,156
Created 3 test queries


In [7]:
print("\n=== Setting up RLM with Tool Calling ===")

# Tool implementations
def tool_grep(pattern):
    """Search for pattern in context."""
    lines = long_context.split('\n')
    matches = [line for line in lines if pattern.lower() in line.lower()]
    if matches:
        return '\n'.join(matches[:5])  # Return top 5 matches
    return "No matches found"

def tool_peek(offset, length):
    """Read text at specific offset."""
    try:
        start = int(offset)
        end = start + int(length)
        return long_context[start:end]
    except:
        return "Invalid offset/length"

def tool_count(pattern):
    """Count occurrences of pattern."""
    count = long_context.lower().count(pattern.lower())
    return f"Found {count} occurrences"

TOOLS = {
    "grep": tool_grep,
    "peek": tool_peek,
    "count": tool_count,
}

# Parse tool calls
TOOL_PATTERN = re.compile(r'TOOL:\s*(\w+)\("([^"]+)"\)')

def parse_tool_call(text):
    """Extract tool call from model output."""
    text = text.strip()
    match = TOOL_PATTERN.search(text)
    if match:
        tool_name = match.group(1)
        args = match.group(2)
        if tool_name in TOOLS:
            return tool_name, args
    return None, None

print("RLM tools configured: grep, peek, count")


=== Setting up RLM with Tool Calling ===
RLM tools configured: grep, peek, count


In [8]:
print("\n=== BASELINE 4: RLM with Tool Calling ===")

def baseline_rlm_tools(context, query, max_steps=5):
    """RLM using grep/peek/count tools recursively."""

    system_prompt = """You are helping answer questions about a long document.
You have access to tools: grep(pattern), peek(offset, length), count(pattern).

To use a tool, output EXACTLY: TOOL: toolname("argument")
Examples:
TOOL: grep("Telemachy")
TOOL: peek("1000, 200")
TOOL: count("Odysseus")

When you have enough information, provide a final answer in plain text without using tools."""

    conversation = f"{system_prompt}\n\nUser: {query}\n\nAssistant:"

    print(f"  Processing: {query[:50]}...")

    for step in range(max_steps):
        response = ask(conversation, max_new_tokens=150)

        if "Assistant:" in response:
            response = response.split("Assistant:")[-1].strip()

        print(f"  Step {step+1}: {response[:80]}...")

        tool_name, tool_arg = parse_tool_call(response)

        if tool_name:
            if tool_name == "peek":
                parts = tool_arg.split(',')
                if len(parts) == 2:
                    result = tool_peek(parts[0].strip(), parts[1].strip())
                else:
                    result = "Invalid peek arguments"
            else:
                result = TOOLS[tool_name](tool_arg)

            print(f"    → Tool {tool_name} executed")
            conversation += f" {response}\n\nTool Result: {result[:200]}\n\nAssistant:"
        else:
            if "answer:" in response.lower():
                parts = response.lower().split("answer:")
                answer = parts[-1].strip()
            else:
                answer = response.strip()
            return answer

    return "Unable to find answer within step limit"

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer_rlm = baseline_rlm_tools(long_context, test_queries[0])
time_rlm = time.time() - start

print(f"\nAnswer: {answer_rlm}")
print(f"Time: {time_rlm:.2f}s\n")


=== BASELINE 4: RLM with Tool Calling ===
Testing: What is the Telemachy?
  Processing: What is the Telemachy?...
  Step 1: The Telemachy refers to the story or narrative that precedes the main action of ...

Answer: The Telemachy refers to the story or narrative that precedes the main action of Homer's epic poem "Odyssey." It involves Telemachus, Odysseus' son, who embarks on a journey to find his absent father. This part of the story is often considered a prelude to the main plot of the Odyssey, where Telemachus grows from a young boy into a man as he seeks adventure and learns about his father's absence. The Telemachy includes events such as the visit of the suitors to the palace and Telemachus' decision to seek news of his father. However, it's important to note that the term "Telemachy" can also refer more broadly to
Time: 22.41s



In [9]:
print("=== BASELINE 5: Map-Reduce Summarization ===")

def baseline_map_reduce(context, query, chunk_size=800):
    """Map-Reduce: summarize chunks then aggregate."""
    words = context.split()
    chunks = [' '.join(words[i:i+chunk_size])
             for i in range(0, len(words), chunk_size)]

    print(f"  Processing {len(chunks)} chunks")

    summaries = []
    for i, chunk in enumerate(chunks[:15]):
        if (i + 1) % 5 == 0:
            print(f"  Summarizing chunk {i+1}/15")

        prompt = f"Summarize this text focusing on '{query}':\n\n{chunk}\n\nSummary:"
        response = ask(prompt, max_new_tokens=100)

        if "Summary:" in response:
            summary = response.split("Summary:")[-1].strip()
        else:
            summary = response.strip()

        summaries.append(summary)

    combined = '\n\n'.join(summaries)
    prompt = f"Based on these summaries:\n{combined}\n\nAnswer: {query}\n\nAnswer:"
    response = ask(prompt, max_new_tokens=200)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

# Test
print(f"Testing: {test_queries[0]}")
start = time.time()
answer_mr = baseline_map_reduce(long_context, test_queries[0])
time_mr = time.time() - start

print(f"\nAnswer: {answer_mr}")
print(f"Time: {time_mr:.2f}s\n")

=== BASELINE 5: Map-Reduce Summarization ===
Testing: What is the Telemachy?
  Processing 324 chunks
  Summarizing chunk 5/15
  Summarizing chunk 10/15
  Summarizing chunk 15/15

Answer: The Telemachy is a section in Homer's epic poem "The Odyssey" that focuses on the journey of Telemachus, Odysseus' son, to Pylos. The passage argues that the "Telemachy" is a distinct section of the larger poem, consisting of Books 8-16, and that it was added to create a unified narrative. Butler suggests that the inclusion of this section was done to maintain continuity with the rest of the epic.

Additional Information:

1. The Telemachy primarily deals with Telemachus' quest to learn about his absent father, Odysseus, and to confront the suitors who have taken over his household.
2. It is believed that the "Telemachy" was added to the original "Odyssey" to provide a more cohesive and complete narrative.
3. The section includes encounters with various gods, such as Athena (Minerva), and it explores t

In [12]:
print("=== BASELINE 6: Hierarchical Summarization ===")

def baseline_hierarchical(context, query, chunk_size=1000, branch_factor=5, max_chunks=50):
    """Hierarchical: multi-level summarization (optimized)."""
    words = context.split()
    current_level = [' '.join(words[i:i+chunk_size])
                    for i in range(0, len(words), chunk_size)]

    if len(current_level) > max_chunks:
        print(f"  Limiting from {len(current_level)} to {max_chunks} chunks")
        current_level = current_level[:max_chunks]

    print(f"  Level 0: {len(current_level)} chunks")
    level = 1

    while len(current_level) > branch_factor and level <= 3:
        next_level = []

        for i in range(0, len(current_level), branch_factor):
            group = current_level[i:i+branch_factor]
            combined = '\n\n'.join(group)

            if len(combined.split()) > 2000:
                combined_words = combined.split()[:2000]
                combined = ' '.join(combined_words)

            prompt = f"Summarize keeping info about '{query}':\n\n{combined}\n\nSummary:"
            response = ask(prompt, max_new_tokens=150)

            if "Summary:" in response:
                summary = response.split("Summary:")[-1].strip()
            else:
                summary = response.strip()

            next_level.append(summary)

        print(f"  Level {level}: {len(next_level)} summaries")
        current_level = next_level
        level += 1

    combined = '\n\n'.join(current_level)
    prompt = f"Based on:\n{combined}\n\nAnswer: {query}\n\nAnswer:"
    response = ask(prompt, max_new_tokens=200)

    if "Answer:" in response:
        answer = response.split("Answer:")[-1].strip()
    else:
        answer = response.strip()

    return answer

print(f"Testing: {test_queries[0]}")
start = time.time()
answer_hier = baseline_hierarchical(long_context, test_queries[0])
time_hier = time.time() - start

print(f"\nAnswer: {answer_hier}")
print(f"Time: {time_hier:.2f}s\n")

=== BASELINE 6: Hierarchical Summarization ===
Testing: What is the Telemachy?
  Limiting from 260 to 50 chunks
  Level 0: 50 chunks
  Level 1: 10 summaries
  Level 2: 2 summaries

Answer: The Telemachy is a Greek mythological story about Odysseus' son Telemachus. After his father Odysseus leaves for the Trojan War, Telemachus grows up without his father's guidance. He becomes concerned about his father's safety and the impending suitors who wish to marry his mother, Penelope. Telemachus decides to seek news of his father and confront the suitors. He embarks on a journey to Pylos and Sparta, where he learns of his father's whereabouts and the dire situation at home. Upon his return, Telemachus faces the suitors and eventually takes action to protect his family and kingdom. This narrative explores themes of parental absence, filial duty, and the challenges faced by a young man seeking to fulfill his responsibilities. The story also highlights the importance of courage, wisdom, and the p

In [13]:
print("\n=== EVALUATING NEW METHODS ===\n")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

results = []

for method_name, method_func in [
    ("RLM-Tools", baseline_rlm_tools),
    ("Map-Reduce", baseline_map_reduce),
    ("Hierarchical", baseline_hierarchical)
]:
    print(f"Evaluating {method_name}...")

    predictions = []
    times = []

    for i, (query, ref) in enumerate(zip(test_queries, reference_answers)):
        print(f"  Query {i+1}/{len(test_queries)}")
        start = time.time()
        pred = method_func(long_context, query)
        times.append(time.time() - start)
        predictions.append(pred)

    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, reference_answers)
    ]
    avg_rouge = np.mean(rouge_scores)
    avg_time = np.mean(times)

    results.append({
        'method': method_name,
        'rouge_l': avg_rouge,
        'avg_time': avg_time
    })

    print(f"  ROUGE-L: {avg_rouge:.3f}")
    print(f"  Avg Time: {avg_time:.2f}s\n")

print("Evaluation complete")


=== EVALUATING NEW METHODS ===

Evaluating RLM-Tools...
  Query 1/3
  Processing: What is the Telemachy?...
  Step 1: The Telemachy is a section or episode in Homer's Odyssey where Telemachus, Odyss...
  Query 2/3
  Processing: Who is Polyphemos?...
  Step 1: Polyphemus lives on an island called Thrinaxandros (or Thrinacia). He is the son...
  Query 3/3
  Processing: What happened to Odysseus during his wanderings?...
  Step 1: To determine what happened to Odysseus during his wanderings, we need to look th...
    → Tool grep executed
  Step 2: It seems there was an error in the initial search. The document does not contain...
  ROUGE-L: 0.149
  Avg Time: 15.34s

Evaluating Map-Reduce...
  Query 1/3
  Processing 324 chunks
  Summarizing chunk 5/15
  Summarizing chunk 10/15
  Summarizing chunk 15/15
  Query 2/3
  Processing 324 chunks
  Summarizing chunk 5/15
  Summarizing chunk 10/15
  Summarizing chunk 15/15
  Query 3/3
  Processing 324 chunks
  Summarizing chunk 5/15
  Summarizing c

In [14]:
print("\n=== RESULTS FOR NEW METHODS ===\n")

print(f"{'Method':<25} {'ROUGE-L':>10} {'Avg Time (s)':>15}")
print("-" * 52)

for r in results:
    print(f"{r['method']:<25} {r['rouge_l']:>10.3f} {r['avg_time']:>15.2f}")

print("-" * 52)

best = max(results, key=lambda x: x['rouge_l'])
fastest = min(results, key=lambda x: x['avg_time'])

print(f"\nBest ROUGE-L: {best['method']} ({best['rouge_l']:.3f})")
print(f"Fastest: {fastest['method']} ({fastest['avg_time']:.2f}s)")

profile()


=== RESULTS FOR NEW METHODS ===

Method                       ROUGE-L    Avg Time (s)
----------------------------------------------------
RLM-Tools                      0.149           15.34
Map-Reduce                     0.097          129.10
Hierarchical                   0.094          164.00
----------------------------------------------------

Best ROUGE-L: RLM-Tools (0.149)
Fastest: RLM-Tools (15.34s)


RAM Used: 6.10 GB

In [15]:
results_dict = {
    'experiment': 'rlm_and_more_methods',
    'model': model_name,
    'context_length': len(long_context.split()),
    'num_queries': len(test_queries),
    'results': results
}

with open('rlm_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

print("\nResults saved to rlm_results.json")


Results saved to rlm_results.json


In [17]:
import json

with open('rlm_results.json', 'r') as f:
    data = json.load(f)
    print(json.dumps(data, indent=2))

{
  "experiment": "rlm_and_more_methods",
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "context_length": 259156,
  "num_queries": 3,
  "results": [
    {
      "method": "RLM-Tools",
      "rouge_l": 0.14870078112545831,
      "avg_time": 15.337470213572184
    },
    {
      "method": "Map-Reduce",
      "rouge_l": 0.09745738132834907,
      "avg_time": 129.10051933924356
    },
    {
      "method": "Hierarchical",
      "rouge_l": 0.09434020283810739,
      "avg_time": 163.99929523468018
    }
  ]
}


In [16]:
print("\n=== SUMMARY ===\n")
print("Tested 3 additional methods:")
print("  - RLM with tool calling (grep, peek, count)")
print("  - Map-Reduce summarization")
print("  - Hierarchical summarization")
print(f"\nContext length: {len(long_context.split()):,} words")
print(f"Evaluated on {len(test_queries)} queries")
print(f"Memory usage: {psutil.virtual_memory().used / 1e9:.2f} GB")


=== SUMMARY ===

Tested 3 additional methods:
  - RLM with tool calling (grep, peek, count)
  - Map-Reduce summarization
  - Hierarchical summarization

Context length: 259,156 words
Evaluated on 3 queries
Memory usage: 6.11 GB
